In [1]:
from scenario_access import load_scenario
import numpy as np
data=load_scenario(32,"",False)

********************************************************
Loading Scenario32 dataset from 


c:\Users\Superuser\Desktop\TT\2025 Papers\deepsense_positon_non_linear\scenario_access.py:83: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  scenario_df.interpolate(method='linear', axis=0, limit_direction='both',inplace=True)


In [2]:
data.head()

,seq_index,unit2_spd_over_grnd_kmph,unit2_altitude,unit2_geo_sep,unit2_mode_fix_type,unit2_pdop,unit2_hdop,unit2_vdop,unit2_interpolated_position,unit2_lat,unit2_lon,bearing,unit1_unit2_distance,unit2_prev_distance,direction,unit1_beam_index,unit1_beam_index_f
0,1,5.820,356.810,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935136,3.949757,49.745416,0.000000,0.0,2,2.0
1,1,5.841,356.812,-27.749,3.0,1.09,0.58,0.92,1.0,33.424312,-111.935134,4.134088,49.753753,0.160271,0.0,2,2.0
2,1,5.862,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,4.318356,49.762605,0.160270,0.0,2,2.0
3,1,6.066,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,4.318356,49.762605,0.000000,0.0,2,2.0
4,1,6.351,356.807,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935131,4.509636,49.768738,0.166255,0.0,4,4.0


In [ ]:
from typing import Any


window_size=10
def create_windows(X, y, window_size=10):
    X_win, y_win = [], []
    for i in range(len(X) - window_size):
        X_win.append(X[i : i + window_size])
        y_win.append(y[i + window_size])
    return np.array(X_win), np.array(y_win)

def generate_sequence_windows(data, window_size=10):
    """
    Generate sequence windows for all groups in the provided DataFrame, then concatenate all into X, y arrays.

    Args:
        data (pd.DataFrame): The input data containing the 'seq_index' column and 'unit1_beam_index'.
        window_size (int): The length of the sliding window.

    Returns:
        X_all (np.ndarray): All input windows concatenated from every group.
        y_all (np.ndarray): All target values concatenated from every group.
    """
    grouped = data.groupby('seq_index')
    X_list = []
    y_list = []
    for seq_idx, group in grouped:
        group = group.reset_index(drop=True)
        X = group.drop(columns=['unit1_beam_index','seq_index']).to_numpy()
        y = group['unit1_beam_index'].to_numpy()
        X_win, y_win = create_windows(X, y, window_size)
        X_list.append(X_win)
        y_list.append(y_win)
    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_all, y_all

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Define the features and target
target_col = 'unit1_beam_index'
feature_cols = [col for col in data.columns if col != target_col and col != 'seq_index']

X_feat, y_feat = generate_sequence_windows(data, window_size)


# Fit Random Forest for feature importance
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_feat.reshape(X_feat.shape[0], -1), y_feat)

# Get feature importances
importances = rf_selector.feature_importances_

# X_feat shape: (n_samples, window_size, n_features_per_time), flattened
n_features_per_time = len(feature_cols)
importances_per_window = importances.reshape(-1, n_features_per_time)
print(importances_per_window.shape)
# Calculate mean importance for each feature across window positions
mean_importances = importances_per_window.mean(axis=0)
print(mean_importances.shape)

feature_importance = sorted(zip(feature_cols, mean_importances), key=lambda x: x[1], reverse=True)

print("Feature importances (descending):")
features = [feat for feat, imp in feature_importance if imp > 0.01]
for feat, imp in feature_importance :
    print(f"{feat}: {imp:.4f}")


# 5 fold split on the grouped (seq-based) windows
n_folds = 5

from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=n_folds)
mean_acc=0
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(data, groups=data["seq_index"])):
    train=data.iloc[train_idx]
    test=data.iloc[test_idx]
    scaler = StandardScaler()
    train.loc[:,features] = scaler.fit_transform(train[features])
    test.loc[:,features] = scaler.transform(test[features])

    X_train, y_train = generate_sequence_windows(train[features+["unit1_beam_index","seq_index"]], window_size)

    
    X_test, y_test = generate_sequence_windows(test[features+["unit1_beam_index","seq_index"]], window_size)

 
    train_seq_indices = data.iloc[train_idx]['seq_index'].unique()
    test_seq_indices = data.iloc[test_idx]['seq_index'].unique()
    print(f"Fold {fold_num+1}:")
    print(f"  Train instances: {len(X_train)}")
    print(f"  Test instances: {len(X_test)}")
    print(f"  Train X shape: {X_train.shape}")
    print(f"  Test X shape: {X_test.shape}")
    print(f"  Train y shape: {y_train.shape}")
    print(f"  Test y shape: {y_test.shape}")
    print(f"  Unique seq_index in train: {train_seq_indices}")
    print(f"  Unique seq_index in test: {test_seq_indices}")
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    # We'll just use the first fold as an example for simplicity:
    # Use the previous loop, so X_train, y_train, X_test, y_test already exist
    # Example: fit and evaluate a Random Forest on the first fold
    clf = RandomForestClassifier(n_estimators=1000, random_state=42)
    clf.fit(X_train.reshape(X_train.shape[0], -1), y_train)

    y_pred = clf.predict(X_test.reshape(X_test.shape[0], -1))
    acc = accuracy_score(y_test, y_pred)
    mean_acc+=acc
    print(f"Random Forest accuracy on fold {fold_num+1}: {acc:.4f}")

print(f"Mean accuracy: {mean_acc/n_folds:.4f}")

(10, 15)
(15,)
Feature importances (descending):
bearing: 0.0212
unit1_unit2_distance: 0.0154
unit2_lat: 0.0133
unit1_beam_index_f: 0.0095
unit2_spd_over_grnd_kmph: 0.0094
unit2_prev_distance: 0.0089
unit2_altitude: 0.0088
unit2_vdop: 0.0031
unit2_pdop: 0.0030
unit2_lon: 0.0025
unit2_hdop: 0.0023
unit2_interpolated_position: 0.0015
unit2_mode_fix_type: 0.0004
direction: 0.0003
unit2_geo_sep: 0.0002
Fold 1:
  Train instances: 2462
  Test instances: 623
  Train X shape: (2462, 10, 3)
  Test X shape: (623, 10, 3)
  Train y shape: (2462,)
  Test y shape: (623,)
  Unique seq_index in train: [ 2  4  6  7  8 10 11 12 13 14 16 17]
  Unique seq_index in test: [1 3 5]
Random Forest accuracy on fold 1: 0.3997
Fold 2:
  Train instances: 2470
  Test instances: 615
  Train X shape: (2470, 10, 3)
  Test X shape: (615, 10, 3)
  Train y shape: (2470,)
  Test y shape: (615,)
  Unique seq_index in train: [ 1  3  5  6  7  8 10 11 12 13 16 17]
  Unique seq_index in test: [ 2  4 14]
Random Forest accuracy o

In [7]:
from typing import Any


window_size=10
def create_windows(X, y, window_size=10):
    X_win, y_win = [], []
    for i in range(len(X) - window_size):
        X_win.append(X[i : i + window_size])
        y_win.append(y[i + window_size])
    return np.array(X_win), np.array(y_win)

def generate_sequence_windows(data, window_size=10):
    """
    Generate sequence windows for all groups in the provided DataFrame, then concatenate all into X, y arrays.

    Args:
        data (pd.DataFrame): The input data containing the 'seq_index' column and 'unit1_beam_index'.
        window_size (int): The length of the sliding window.

    Returns:
        X_all (np.ndarray): All input windows concatenated from every group.
        y_all (np.ndarray): All target values concatenated from every group.
    """
    grouped = data.groupby('seq_index')
    X_list = []
    y_list = []
    for seq_idx, group in grouped:
        group = group.reset_index(drop=True)
        X = group.drop(columns=['unit1_beam_index','seq_index']).to_numpy()
        y = group['unit1_beam_index'].to_numpy()
        X_win, y_win = create_windows(X, y, window_size)
        X_list.append(X_win)
        y_list.append(y_win)
    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_all, y_all

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Define the features and target
target_col = 'unit1_beam_index'
feature_cols = [col for col in data.columns if col != target_col and col != 'seq_index']

X_feat, y_feat = generate_sequence_windows(data, window_size)


# Fit Random Forest for feature importance
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_feat.reshape(X_feat.shape[0], -1), y_feat)

# Get feature importances
importances = rf_selector.feature_importances_

# X_feat shape: (n_samples, window_size, n_features_per_time), flattened
n_features_per_time = len(feature_cols)
importances_per_window = importances.reshape(-1, n_features_per_time)
print(importances_per_window.shape)
# Calculate mean importance for each feature across window positions
mean_importances = importances_per_window.mean(axis=0)
print(mean_importances.shape)

feature_importance = sorted(zip(feature_cols, mean_importances), key=lambda x: x[1], reverse=True)

print("Feature importances (descending):")
features = [feat for feat, imp in feature_importance if imp > 0.01]
for feat, imp in feature_importance :
    print(f"{feat}: {imp:.4f}")


# 5 fold split on the grouped (seq-based) windows
n_folds = 5

from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=n_folds)
mean_acc=0
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(data, groups=data["seq_index"])):
    train=data.iloc[train_idx]
    test=data.iloc[test_idx]
    scaler = StandardScaler()
    train.loc[:,features] = scaler.fit_transform(train[features])
    test.loc[:,features] = scaler.transform(test[features])

    X_train, y_train = generate_sequence_windows(train[features+["unit1_beam_index","seq_index"]], window_size)

    
    X_test, y_test = generate_sequence_windows(test[features+["unit1_beam_index","seq_index"]], window_size)

 
    train_seq_indices = data.iloc[train_idx]['seq_index'].unique()
    test_seq_indices = data.iloc[test_idx]['seq_index'].unique()
    print(f"Fold {fold_num+1}:")
    print(f"  Train instances: {len(X_train)}")
    print(f"  Test instances: {len(X_test)}")
    print(f"  Train X shape: {X_train.shape}")
    print(f"  Test X shape: {X_test.shape}")
    print(f"  Train y shape: {y_train.shape}")
    print(f"  Test y shape: {y_test.shape}")
    print(f"  Unique seq_index in train: {train_seq_indices}")
    print(f"  Unique seq_index in test: {test_seq_indices}")
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    # We'll just use the first fold as an example for simplicity:
    # Use the previous loop, so X_train, y_train, X_test, y_test already exist
    # Example: fit and evaluate a Random Forest on the first fold
    clf = RandomForestClassifier(n_estimators=1000, random_state=42)
    clf.fit(X_train.reshape(X_train.shape[0], -1), y_train)

    y_pred = clf.predict(X_test.reshape(X_test.shape[0], -1))
    acc = accuracy_score(y_test, y_pred)
    mean_acc+=acc
    print(f"Random Forest accuracy on fold {fold_num+1}: {acc:.4f}")

print(f"Mean accuracy: {mean_acc/n_folds:.4f}")

(10, 15)
(15,)
Feature importances (descending):
bearing: 0.0212
unit1_unit2_distance: 0.0154
unit2_lat: 0.0133
unit1_beam_index_f: 0.0095
unit2_spd_over_grnd_kmph: 0.0094
unit2_prev_distance: 0.0089
unit2_altitude: 0.0088
unit2_vdop: 0.0031
unit2_pdop: 0.0030
unit2_lon: 0.0025
unit2_hdop: 0.0023
unit2_interpolated_position: 0.0015
unit2_mode_fix_type: 0.0004
direction: 0.0003
unit2_geo_sep: 0.0002
Fold 1:
  Train instances: 2462
  Test instances: 623
  Train X shape: (2462, 10, 3)
  Test X shape: (623, 10, 3)
  Train y shape: (2462,)
  Test y shape: (623,)
  Unique seq_index in train: [ 2  4  6  7  8 10 11 12 13 14 16 17]
  Unique seq_index in test: [1 3 5]
Random Forest accuracy on fold 1: 0.3997
Fold 2:
  Train instances: 2470
  Test instances: 615
  Train X shape: (2470, 10, 3)
  Test X shape: (615, 10, 3)
  Train y shape: (2470,)
  Test y shape: (615,)
  Unique seq_index in train: [ 1  3  5  6  7  8 10 11 12 13 16 17]
  Unique seq_index in test: [ 2  4 14]
Random Forest accuracy o